# Day 35: Quantization Bit Width Sweep: Quality vs Compression

**Layer:** Implementation | **Prerequisite:** Previous days


## Concept Overview

Systematically sweep quantization bit widths from INT2 to INT8 and plot the Pareto curve of output quality vs memory compression. This reveals why INT4 is the common production choice — it sits at the knee of the quality-compression curve.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')


## 1. Quantization Bit Width Sweep


In [ ]:
def load_model_and_quantize(bits):
    torch.manual_seed(42)
    model = nn.Sequential(
        nn.Linear(512, 1024), nn.GELU(),
        nn.Linear(1024, 512), nn.GELU(),
        nn.Linear(512, 256)
    )
    # Fake-quantize each linear weight
    for module in model.modules():
        if isinstance(module, nn.Linear):
            qmax = 2**(bits-1) - 1
            scale = module.weight.data.abs().max() / qmax
            W_q = torch.clamp(torch.round(module.weight.data / scale), -qmax, qmax)
            module.weight.data = W_q * scale
    return model

torch.manual_seed(42)
x_test = torch.randn(100, 512)

# Full precision reference
ref_model = nn.Sequential(
    nn.Linear(512, 1024), nn.GELU(),
    nn.Linear(1024, 512), nn.GELU(),
    nn.Linear(512, 256)
)
ref_out = ref_model(x_test)

results = []
for bits in [2, 3, 4, 5, 6, 7, 8, 16]:
    q_model = load_model_and_quantize(bits)
    q_out = q_model(x_test)
    rel_err = (ref_out - q_out).abs().mean() / ref_out.abs().mean()
    # Memory savings
    mem_ratio = 16 / bits
    results.append({'bits': bits, 'rel_error': rel_err.item(), 'mem_ratio': mem_ratio})
    print(f'{bits:>4} bits: rel_error={rel_err.item():.4f}, mem={1/mem_ratio:.2f}x (vs FP16)')


## 2. Quality vs Compression Pareto Curve


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
bits_list = [r['bits'] for r in results]
errors = [r['rel_error'] for r in results]
ratios = [1/r['mem_ratio'] for r in results]  # fraction of FP16 memory

ax.scatter(ratios, errors, s=100, zorder=5)
ax.plot(ratios, errors, 'b--', alpha=0.5)
for r in results:
    ax.annotate(f'{r["bits"]}b', (1/r['mem_ratio'], r['rel_error']),
                textcoords='offset points', xytext=(5,5))
ax.set_xlabel('Memory fraction (vs FP16)')
ax.set_ylabel('Relative output error')
ax.set_title('Quality vs Compression: Bit Width Pareto Curve')
ax.grid(True)
ax.set_yscale('log')
plt.savefig('quant_pareto.png', dpi=100, bbox_inches='tight')
plt.show()
print('INT4 is typically the Pareto-optimal point: 4x compression, ~1-2% error')


## Experiments

1. Add perplexity evaluation: load a small dataset and compute perplexity at each bit width.
2. Try mixed-precision: use INT8 for early layers and INT4 for later layers. Does this improve the Pareto curve?
3. Compare AWQ-style scaling before quantization — how much does it shift the error curve?


## Key Takeaways

- See concept overview above for the key points.
- Day 35 complete.
